# Find LSSTCamSources in all bands add mjd and deepfield


---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-11
- **Last update:** 2026-07-11


## Goal

For each Deep Drilling Field (DDF) and each LSST band, find bright point-like
sources (stars) with `17 <= mag <= 22`, cross-match repeated detections of the
same physical object across visits (same RA/Dec within `MATCH_RADIUS_ARCSEC`),
keep objects with at least `MIN_VISITS_PER_BAND` visits in that band, and
compute the relative flux scatter `sigma_F/F` (expressed in mmag) using
`psfFlux`. Processing is done **band by band, DDF by DDF** to control memory
usage: each chunk is loaded, reduced to per-object statistics, written to
disk, then freed before moving to the next chunk.

Expectation: the relative photometric scatter should be largest in **u** and
**y**, the bands most affected by atmospheric extinction variability
(Rayleigh + aerosols in u, water vapour/aerosols in y) and lower system
throughput.


## 1. Imports

In [ ]:
import gc
import logging
import os
import platform
import re
import resource
import sys
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## 2. Logging

In [ ]:
# 1. Recuperer le logger racine (ou creez un logger specifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Definir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Eviter la duplication des handlers si la cellule est executee plusieurs fois
if not log.handlers:
    # 4. Creer un handler qui ecrit vers la sortie standard (capturee par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Definir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour verifier que ca fonctionne
log.info("Le logging est configure et fonctionne dans le notebook !")

## 3. Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "FindLSSTCamSources_01b"


# ── Output figures ──────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)


# ── Output data ─────────────────────────────────────
DIR_DATA_OUT = f"./data_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
log.info("Data output directory: %s", DIR_DATA_OUT)


# -- Butler --------------------------------------------------------------
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# All LSST bands to loop over (band-by-band processing to control memory)
BANDS = ["u", "g", "r", "i", "z", "y"]
BANDS_CMAP = {"u": "Purples", "g": "Greens", "r": "Reds", "i": "YlOrBr", "z": "pink_r", "y": "bone_r"}
BANDS_COLOR = {
    "u": "blueviolet",
    "g": "limegreen",
    "r": "red",
    "i": "darkorange",
    "z": "chocolate",
    "y": "saddlebrown",
}

# Marker style per DDF (used in the mmag-deviation-vs-MJD light-curve plots)
DDF_MARKERS = {
    "COSMOS": "o",
    "XMM-LSS": "s",
    "ECDFS": "^",
    "ELAIS-S1": "D",
    "EDFS": "v",
}
DDF_MARKER_DEFAULT = "x"  # fallback for any DDF not listed above

# -- Cross-match search radius ----------------------------------------------
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

# -- Magnitude window for bright stars --------------------------------------
MAG_MIN = 17.0
MAG_MAX = 19.5

# -- Minimum number of visits per band for an object to be kept -------------
MIN_VISITS_PER_BAND = {"u": 20, "g": 50, "r": 50, "i": 50, "z": 50, "y": 20}

# -- Star/galaxy separation --------------------------------------------------
EXTENDEDNESS_MAX = 0.5  # base_ClassificationExtendedness: 0 = point source, 1 = extended
SIZEEXTENDEDNESS_MAX = 0.05

# -- Flux definition used for the variability analysis -----------------------
FLUX_COL = "psfFlux"
FLUXERR_COL = "psfFluxErr"

# nJy -> AB magnitude zero point (LSST calibrated fluxes are stored in nanojansky)
ABMAG_ZP_NJY = 31.4

# -- Output ------------------------------------------------------------------
OUTPUT_DIR = "output_objectstats"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
log.debug(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

# All columns available on the source table (kept for reference / provenance,
# e.g. if you later want aperture photometry or pixel positions too).
SRC_COLUMNS_FULL = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    "extendedness",
    "sizeExtendedness",
    "apFlux_12_0_flag",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    "extendedness_flag",
    "sizeExtendedness_flag",
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

# Only the columns actually consumed by clean_sources / add_ab_mag /
# assign_object_ids / summarize_objects. SRC_COLUMNS_FULL above loads ~54
# columns per visit (aperture fluxes at several radii, pixel x/y, local
# background, ...) that are never touched downstream -- for bands with many
# thousands of refs (g/r/i/z have 2-4x more visits than u) this inflates the
# per-visit raw table size for no benefit and is the main lever left to bring
# memory down further, on top of the per-visit filtering already in place.
SRC_COLUMNS_MINIMAL = [
    "coord_ra",
    "coord_dec",
    "sourceId",
    "visit",
    "detector",
    "band",
    FLUX_COL,
    FLUXERR_COL,
    "extendedness",
    "sizeExtendedness",
    "extendedness_flag",
    "sizeExtendedness_flag",
    "sky_source",
]

USE_MINIMAL_COLUMNS = True  # set False to fall back to SRC_COLUMNS_FULL
SRC_COLUMNS = SRC_COLUMNS_MINIMAL if USE_MINIMAL_COLUMNS else SRC_COLUMNS_FULL

log.info(f"Butler configuration done. Loading {len(SRC_COLUMNS)} columns per visit.")

In [ ]:
# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

log.info(f"DDF Selected {list(DEEP_FIELDS.keys())}")

## 4. Helper functions

In [ ]:
# ── savefig: PDF + PNG ───────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

## 5. Process

### 5.1  Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

### 5.2 Find  Which visits

In [ ]:
visits = butler.registry.queryDimensionRecords("visit")

target_names = sorted({v.target_name for v in visits if v.target_name is not None})
print(target_names)

### 5.3 Handle the case multi-target names

In [ ]:
ddf_keywords = ["cosmos", "ecdfs", "cdfs", "xmm", "elaiss1", "elais", "edfs"]


def is_ddf(name):
    n = name.lower()
    return any(k in n for k in ddf_keywords)


ddf_names = sorted({name for name in target_names if is_ddf(name)})

for n in ddf_names:
    print(n)

In [ ]:
def normalize_ddf(name):
    n = name.lower()
    if "cosmos" in n:
        return "COSMOS"
    if "ecdfs" in n or "cdfs" in n:
        return "ECDFS"
    if "xmm" in n:
        return "XMM-LSS"
    if "elaiss1" in n or "elais" in n:
        return "ELAIS-S1"
    if "edfs" in n:
        return "EDFS"
    return None


unique_ddf = sorted({normalize_ddf(name) for name in target_names if normalize_ddf(name) is not None})

print(unique_ddf)

In [ ]:
counter = Counter(normalize_ddf(v.target_name) for v in visits if normalize_ddf(v.target_name))

for k, v in counter.items():
    print(k, v)

### 5.4 Retrieve all Visits in all DDF

In [ ]:
visits = list(butler.registry.queryDimensionRecords("visit"))

ddf_visits = {"COSMOS": [], "XMM-LSS": [], "EDFS": [], "ECDFS": [], "ELAIS-S1": []}

for v in visits:
    name = normalize_ddf(v.target_name)
    if name:
        ddf_visits[name].append(v.id)

# Not working v.band does not exist.
# also index visits by band, needed below to query the source dataset band by band
# visits_band = {v.id: v.band for v in visits}

### 5.5 Auto-discover the object-table dataset type

We probe the registry so the notebook is collection-agnostic. Here we actually
need the **per-visit source table** (not the coadd object table), since we
want repeat single-epoch measurements to build the light curves.


In [ ]:
def dataset_type_exists(butler_obj, name):
    """Return True if `name` is a registered dataset type in this Butler."""
    try:
        butler_obj.registry.getDatasetType(name)
        return True
    except Exception:
        return False

In [ ]:
# Prioritised list of candidate per-visit source-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable",
    "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry (diagnostic only)
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        log.info(f"Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

In [ ]:
# Quick schema probe on a single DDF/band chunk before running the full loop
probe_refsall = list(butler.query_datasets(SRC_DATASET, where="band = 'y'"))
probe_refs = [r for r in probe_refsall if r.dataId["visit"] in ddf_visits["COSMOS"]]

# to get the whole list of columns
# df_probe = butler.get(probe_refs[0])
# to retrieve a limited number of columns
df_probe = butler.get(probe_refs[0], parameters={"columns": SRC_COLUMNS})
if not isinstance(df_probe, pd.DataFrame):
    df_probe = df_probe.to_pandas()
log.info(f"Probe source table: {len(df_probe)} rows, {len(df_probe.columns)} columns")
log.info(f"Columns of sources dataframe : {df_probe.columns.tolist()}")

# sanity check on the flux unit: LSST calibrated fluxes are in nanojansky (nJy).
# psfFlux for 17-22 mag stars should roughly be in the range 1e3 .. 1e6 nJy.
log.info(f"psfFlux range in probe: {np.nanpercentile(df_probe['psfFlux'], [1, 50, 99])}")

del df_probe, probe_refsall, probe_refs
gc.collect()

## 6. Helper functions: cleaning, magnitudes, object cross-match, per-object statistics

The cross-match strategy ("which sources belong to the same physical star")
is a **friends-of-friends (single-linkage) clustering** in the tangent plane:

1. Project `(ra, dec)` onto a local tangent plane in arcsec:
   `x = ra * cos(dec) * 3600`, `y = dec * 3600` (fine approximation over a
   DDF field of only a few degrees).
2. Build a `scipy.spatial.cKDTree` on `(x, y)` and find all pairs of
   detections closer than `MATCH_RADIUS_ARCSEC` (`tree.query_pairs`).
3. Build a sparse adjacency graph from those pairs and extract its
   **connected components** (`scipy.sparse.csgraph.connected_components`).
   Each connected component = one physical object; all detections in it
   (possibly from many different visits) get the same `object_id`.

This is equivalent to what `astropy.coordinates.match_to_catalog_sky` would
give if you picked one reference visit, but it does not require choosing a
reference catalogue and it is robust to a source being missing in some
visits.


In [ ]:
def clean_sources(df, extendedness_max=EXTENDEDNESS_MAX, sizeextendedness_max=SIZEEXTENDEDNESS_MAX):
    """Basic quality cuts: remove sky-background sources, flagged
    extendedness/size measurements, non-positive fluxes, and extended
    (galaxy-like) sources. Returns a filtered copy.
    """
    sel = np.ones(len(df), dtype=bool)

    if "sky_source" in df.columns:
        sel &= ~df["sky_source"].to_numpy(dtype=bool)

    if "extendedness_flag" in df.columns:
        sel &= ~df["extendedness_flag"].to_numpy(dtype=bool)

    if "sizeExtendedness_flag" in df.columns:
        sel &= ~df["sizeExtendedness_flag"].to_numpy(dtype=bool)

    if "extendedness" in df.columns:
        ext = df["extendedness"].to_numpy(dtype=float)
        sel &= np.isfinite(ext) & (ext <= extendedness_max)

    if "sizeExtendedness" in df.columns:
        ext = df["sizeExtendedness"].to_numpy(dtype=float)
        sel &= np.isfinite(ext) & (ext <= sizeextendedness_max)

    flux = df[FLUX_COL].to_numpy(dtype=float)
    fluxerr = df[FLUXERR_COL].to_numpy(dtype=float)
    sel &= np.isfinite(flux) & (flux > 0) & np.isfinite(fluxerr) & (fluxerr > 0)

    return df.loc[sel].reset_index(drop=True)


def add_ab_mag(df, flux_col=FLUX_COL, mag_col="mag", zp=ABMAG_ZP_NJY):
    """Add an AB magnitude column computed from a calibrated flux in nJy."""
    df = df.copy()
    df[mag_col] = -2.5 * np.log10(df[flux_col].to_numpy(dtype=float)) + zp
    return df


def assign_object_ids(df, ra_col="coord_ra", dec_col="coord_dec", radius_arcsec=MATCH_RADIUS_ARCSEC):
    """Group repeated detections of the same physical object using a
    friends-of-friends clustering within `radius_arcsec` on the sky.
    Adds an integer `object_id` column (unique within this dataframe only).
    """
    df = df.copy()
    n = len(df)
    if n == 0:
        df["object_id"] = np.array([], dtype=int)
        return df

    ra = df[ra_col].to_numpy(dtype=float)
    dec = df[dec_col].to_numpy(dtype=float)
    dec_rad = np.deg2rad(dec)

    # local tangent-plane approximation, valid for ~deg-scale DDF fields
    x = ra * np.cos(dec_rad) * 3600.0  # arcsec
    y = dec * 3600.0  # arcsec

    coords = np.column_stack([x, y])
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=radius_arcsec, output_type="ndarray")

    if len(pairs) > 0:
        row, col = pairs[:, 0], pairs[:, 1]
        graph = coo_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
        n_components, labels = connected_components(graph, directed=False)
    else:
        labels = np.arange(n)
        n_components = n

    df["object_id"] = labels
    log.debug(f"assign_object_ids: {n} detections -> {n_components} distinct objects")
    return df


def summarize_objects(
    df,
    band,
    flux_col=FLUX_COL,
    fluxerr_col=FLUXERR_COL,
    mag_col="mag",
    min_visits=MIN_VISITS_PER_BAND,
    mag_min=MAG_MIN,
    mag_max=MAG_MAX,
):
    """
    Aggregate per-detection rows (already tagged with `object_id`) into
    one row per object.

    For each object:
      - compute the median flux as a robust reference flux
      - measure the flux scatter around the median using the MAD estimator
      - estimate the expected photometric noise from flux uncertainties

    The measured relative scatter is:

        sigmaF_over_F = MAD(flux) / median(flux)

    where:

        MAD(flux) = 1.4826 * median(|flux_i - median(flux)|)

    The corresponding magnitude scatter is expressed in mmag.
    """

    df = df.copy()

    # Group detections by object
    g = df.groupby("object_id", sort=False)

    # Number of visits per object
    n_visits = g.size().rename("n_visits")

    # Mean sky position
    ra_mean = g["coord_ra"].mean().rename("ra")
    dec_mean = g["coord_dec"].mean().rename("dec")

    # Flux statistics
    flux_mean = g[flux_col].mean().rename("flux_mean")
    flux_median = g[flux_col].median().rename("flux_median")

    # Robust flux scatter around the median (MAD estimator)
    flux_mad = g[flux_col].apply(lambda x: 1.4826 * np.median(np.abs(x - np.median(x)))).rename("flux_mad")

    # Magnitude statistics
    mag_median = g[mag_col].median().rename("mag_median")

    # Expected relative photometric noise from measurement uncertainties
    relerr = df[fluxerr_col] / df[flux_col]

    photnoise_med = relerr.groupby(df["object_id"], sort=False).median().rename("sigmaF_over_F_phot")

    # Merge all object-level quantities
    out = pd.concat(
        [
            n_visits,
            ra_mean,
            dec_mean,
            flux_mean,
            flux_median,
            flux_mad,
            mag_median,
            photnoise_med,
        ],
        axis=1,
    )

    out = out.reset_index()

    # Relative measured flux scatter
    out["sigmaF_over_F_meas"] = out["flux_mad"] / out["flux_median"]

    # Convert relative flux scatter to magnitude scatter (mmag)
    # dm = -2.5/log(10) * dF/F
    mmag_factor = 2.5 / np.log(10.0) * 1000.0

    out["mmag_meas"] = mmag_factor * out["sigmaF_over_F_meas"]

    out["mmag_phot"] = mmag_factor * out["sigmaF_over_F_phot"]

    # Select objects with enough visits and within the magnitude range
    sel = (out["n_visits"] >= min_visits[band]) & out["mag_median"].between(mag_min, mag_max)

    out = out.loc[sel].reset_index(drop=True)

    return out

## 7. Main loop: band by band, DDF by DDF

For each band we first list the source-table refs for that band once, then
loop over DDFs (filtering the refs list by visit id). Each DDF/band chunk is
loaded, cleaned, magnitude-cut, cross-matched into objects, reduced to
per-object statistics, tagged with `ddf`/`band`, and appended to a
per-band accumulator. The raw per-visit dataframe is deleted and
garbage-collected before moving on, so memory stays bounded by the size of
one DDF/band chunk rather than the whole survey.


In [ ]:
TMP_DIR = os.path.join(DIR_DATA_OUT, "_tmp_chunks")
os.makedirs(TMP_DIR, exist_ok=True)

all_band_results = {}
all_band_lc_results = {}

# Memory monitoring: current RSS (psutil, a snapshot) and true PEAK RSS since
# process start (resource.ru_maxrss, monotonic -- this is what the OS/OOM
# killer actually reacts to, unlike a snapshot which can miss a transient
# spike that happens *between* two log lines, e.g. during a big pd.concat).
try:
    import psutil

    _proc = psutil.Process()

    def _rss_gb():
        return _proc.memory_info().rss / 1e9
except ImportError:

    def _rss_gb():
        return float("nan")


# ru_maxrss is in KB on Linux but in BYTES on macOS.
_RSS_UNIT = 1e9 if platform.system() == "Darwin" else 1e6


def _peak_rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / _RSS_UNIT


# loop on bands
for band in BANDS:
    log.info(f"=== Band '{band}': listing source-table refs ===")
    refsall = list(butler.query_datasets(SRC_DATASET, where="band = :b", bind={"b": band}))
    log.info(
        f"Band '{band}': {len(refsall)} refs total (all DDFs), "
        f"RSS={_rss_gb():.2f} GB, peak={_peak_rss_gb():.2f} GB"
    )

    band_chunks = []
    band_lc_chunks = []

    # loop on DDF
    for ddf_name in unique_ddf:
        visit_ids = set(ddf_visits[ddf_name])
        refs = [r for r in refsall if r.dataId["visit"] in visit_ids]
        if not refs:
            log.info(f"  [{band}/{ddf_name}] no refs, skipping")
            continue

        # -----------------------------------------------------------------
        # Stream filtered per-visit chunks straight to a temp parquet file
        # instead of accumulating pa.Table / pandas.DataFrame objects in a
        # Python list. Creating hundreds of small pyarrow Tables per DDF
        # (previous version) leaks native memory into pyarrow's internal
        # allocator pool, which never returns it to the OS -- that's the
        # real cause of the monotonically climbing peak RSS (2.18 -> 11.93
        # GB across bands). Writing to disk keeps at most ONE ref's data +
        # the ParquetWriter's row-group buffer resident at a time, and we
        # do exactly ONE clean read-back per DDF at the end.
        # -----------------------------------------------------------------
        tmp_path = os.path.join(TMP_DIR, f"{band}_{ddf_name}.parquet")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        writer = None
        n_raw_total = 0
        n_filtered_total = 0

        # loop on all sources in ddf for that band
        for i, ref in enumerate(refs):
            d = butler.get(ref, parameters={"columns": SRC_COLUMNS})
            if not isinstance(d, pd.DataFrame):
                d = d.to_pandas()

            n_raw_total += len(d)

            # select only good point-like sources
            d = clean_sources(d)

            # calculate the magnitude on psf
            d = add_ab_mag(d)

            # filter the sources in the magnitude range
            d = d.loc[d["mag"].between(MAG_MIN, MAG_MAX)].reset_index(drop=True)

            # downcast to shrink the memory footprint of the written chunks
            for col in (FLUX_COL, FLUXERR_COL, "mag", "coord_ra", "coord_dec"):
                if col in d.columns:
                    d[col] = d[col].astype("float32")

            # add the the mjd columns
            # add the mjd columns
            list_of_visits = d["visit"].unique()

            if len(list_of_visits) > 0:
                records = butler.registry.queryDimensionRecords(
                    "visit", where=f"visit IN ({','.join(map(str, list_of_visits))})"
                )

                visit_to_mjd = {r.id: r.timespan.begin.mjd for r in records}

                d["mjd"] = d["visit"].map(visit_to_mjd).astype("float32")
            else:
                d["mjd"] = np.nan
                d["mjd"] = d["mjd"].astype("float32")

            # there should be No missing mjd
            # n_missing = d["mjd"].isna().sum()
            # log.info(f"Missing MJD for {n_missing} rows")

            # add the ddf name
            d["ddf_name"] = ddf_name

            # save on disk the sources from that (band,ddf)
            if len(d) > 0:
                table = pa.Table.from_pandas(d, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(tmp_path, table.schema)
                writer.write_table(table)
                n_filtered_total += len(d)
                del table

            del d
            # if i % 100 == 0:
            if i % 50 == 0:
                gc.collect()

        if writer is not None:
            writer.close()
        del writer
        gc.collect()
        pa.default_memory_pool().release_unused()  # force Arrow to actually free its pool

        if n_filtered_total == 0:
            log.info(f"  [{band}/{ddf_name}] {n_raw_total} raw rows -> 0 after cuts, skipping")
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            continue

        # single controlled read-back of the already-filtered (small) data
        df = pq.read_table(tmp_path).to_pandas()
        df["ddf_name"] = df["ddf_name"].astype("category")

        os.remove(tmp_path)
        gc.collect()

        # associates the different sources each to one object
        df = assign_object_ids(df)

        # calculate the quantity of interest at the object level
        obj = summarize_objects(df, band)
        obj["ddf"] = ddf_name
        obj["band"] = band

        # -------------------------------------------------------------
        # Per-detection light-curve deviation from the per-object MEDIAN
        # flux, expressed in mmag: dmmag = -2.5*log10(F / median(F)).
        # Restricted to detections of objects retained in `obj` (i.e.
        # passing the n_visits/magnitude cuts), so this stays small.
        # -------------------------------------------------------------
        kept_ids = obj["object_id"].to_numpy()
        lc = df.loc[df["object_id"].isin(kept_ids), ["object_id", "mjd", FLUX_COL, "ddf_name"]].copy()
        flux_median = df.groupby("object_id")[FLUX_COL].median().rename("flux_median")
        lc = lc.merge(flux_median, on="object_id", how="left")
        lc["dmmag"] = (
            -2.5
            * np.log10(lc[FLUX_COL].to_numpy(dtype=float) / lc["flux_median"].to_numpy(dtype=float))
            * 1000.0
        )
        lc["band"] = band
        lc["ddf"] = ddf_name
        lc = lc.drop(columns=[FLUX_COL, "flux_median", "ddf_name"])
        band_lc_chunks.append(lc)
        del lc, flux_median, kept_ids

        log.info(
            f"  [{band}/{ddf_name}] {n_raw_total} raw rows -> {len(df)} after mag/quality cuts "
            f"-> {len(obj)} objects with >= {MIN_VISITS_PER_BAND[band]} visits, "
            f"RSS={_rss_gb():.2f} GB, peak={_peak_rss_gb():.2f} GB"
        )

        band_chunks.append(obj)

        del df, obj
        gc.collect()
        pa.default_memory_pool().release_unused()

    if band_chunks:
        df_band = pd.concat(band_chunks, ignore_index=True)
    else:
        df_band = pd.DataFrame()

    out_path = os.path.join(DIR_DATA_OUT, f"objectstats_band_{band}.parquet")
    df_band.to_parquet(out_path, index=False)
    log.info(f"Band '{band}': {len(df_band)} objects (all DDFs) written to {out_path}")

    all_band_results[band] = df_band

    if band_lc_chunks:
        df_band_lc = pd.concat(band_lc_chunks, ignore_index=True)
    else:
        df_band_lc = pd.DataFrame()

    lc_out_path = os.path.join(DIR_DATA_OUT, f"lcdeviation_band_{band}.parquet")
    df_band_lc.to_parquet(lc_out_path, index=False)
    log.info(
        f"Band '{band}': {len(df_band_lc)} detections (mmag deviation from median) written to {lc_out_path}"
    )

    all_band_lc_results[band] = df_band_lc

    del refsall, band_chunks, df_band, band_lc_chunks, df_band_lc
    gc.collect()

## 8. Combine all bands and inspect the results

In [ ]:
df_all = pd.concat(
    [df for df in all_band_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total objects across all bands/DDFs: {len(df_all)}")

df_all.to_csv(os.path.join(DIR_DATA_OUT, "objectstats_allbands.csv"), index=False)
df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

## 8b. Combine per-detection light-curve deviations across bands

For each retained object, `dmmag` is the deviation of an individual visit's
`psfFlux` from that object's **median** flux across all its visits in that
band, expressed in mmag: `dmmag = -2.5*log10(F / median(F)) * 1000`. This
is the quantity plotted against MJD below (section 9.4).


In [ ]:
df_lc_all = pd.concat(
    [df for df in all_band_lc_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total per-detection light-curve deviations across all bands/DDFs: {len(df_lc_all)}")

df_lc_all.to_parquet(os.path.join(DIR_DATA_OUT, "lcdeviation_allbands.parquet"), index=False)
df_lc_all.head()

## 9. Plots

### 9.1  Plot: relative photometric scatter (mmag) per band : Box plot Non outliers

Boxplot of `mmag_meas` (measured scatter, using `psfFlux`) grouped by band,
in the standard LSST band order `u, g, r, i, z, y`. We expect the largest
scatter in **u** and **y**.


In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(data, labels=band_order, showfliers=False)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band.png"), dpi=150)
# plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band.pdf"))

savefig(fig, "mmag_scatter_allsrc_perband")

plt.show()

### 9.2  Plot: relative photometric scatter (mmag) per band -- with outliers

In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
flier_props_71 = dict(
    marker="o",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="gray",
    markeredgewidth=1.0,
    alpha=0.6,
)
ax.boxplot(data, tick_labels=band_order, showfliers=True, flierprops=flier_props_71)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.0, 200.0)
plt.tight_layout()


savefig(fig, "mmag_scatter_allsrc_perband")

plt.show()

In [ ]:
sub.head()

### 9.1  Plot: relative photometric scatter (mmag) vs magnitude

In [ ]:
# Cross-check: measured scatter vs. photon-noise-only expectation, per band
fig, ax = plt.subplots(figsize=(8, 6))
for b in band_order:
    sub = df_all.loc[df_all["band"] == b]
    ax.scatter(sub["mag_median"], sub["mmag_meas"], s=10, color=BANDS_COLOR[b], alpha=0.4, label=b)
ax.set_xlabel("median magnitude")
ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")
ax.set_yscale("log")
ax.legend(markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig(fig, "mmag_scatter_allsrc_allbands")
plt.show()

### 9.4  Light curves : Flux deviation from the per-object median (mmag) vs MJD, per band

For every retained object and every one of its visits, we plot the deviation
of that visit's `psfFlux` from the object's own median flux (in mmag) against
the visit MJD, for **all objects at once**, one panel per band:

- **color** encodes the **band** (using `BANDS_COLOR`)
- **marker** encodes the **DDF** (using `DDF_MARKERS`)
- the **bottom** x-axis is MJD, the **top** x-axis shows the calendar date
  (`YYYY-MM-DD`)


In [ ]:
def add_top_date_axis(ax, mjd_min, mjd_max, n_ticks=6):
    """Add a secondary x-axis on top of *ax* showing calendar dates
    (YYYY-MM-DD) at evenly spaced MJD positions between mjd_min/mjd_max.
    """
    ticks_mjd = np.linspace(mjd_min, mjd_max, n_ticks)
    ticks_date = Time(ticks_mjd, format="mjd").to_value("iso", subfmt="date")
    secax = ax.secondary_xaxis("top")
    secax.set_xticks(ticks_mjd)
    secax.set_xticklabels(ticks_date, rotation=45, ha="left")
    secax.set_xlabel("Date")
    return secax


# configuration of the figure
band_order_lc = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_lc_all["band"].unique()]
n_bands_lc = len(band_order_lc)
ncols_lc = 2
nrows_lc = int(np.ceil(n_bands_lc / ncols_lc))

mjd_min_all = df_lc_all["mjd"].min()
mjd_max_all = df_lc_all["mjd"].max()


YAXIS_MAX_MAG = 500.0

# create the figure
fig, axes = plt.subplots(nrows_lc, ncols_lc, figsize=(16, 3.2 * nrows_lc), sharex=True)
axes = np.atleast_1d(axes).ravel()

# loop on bands
for i, band in enumerate(band_order_lc):
    ax = axes[i]
    sub_band = df_lc_all.loc[df_lc_all["band"] == band]

    # loop on ddf
    for ddf_name in unique_ddf:
        sub = sub_band.loc[sub_band["ddf"] == ddf_name]
        if len(sub) == 0:
            continue
        marker = DDF_MARKERS.get(ddf_name, DDF_MARKER_DEFAULT)
        ax.scatter(
            sub["mjd"],
            sub["dmmag"],
            s=8,
            alpha=0.35,
            color=BANDS_COLOR[band],
            marker=marker,
            linewidths=0,
            label=ddf_name,
        )

    ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)

    # clip y-range to the 0.5-99.5 percentile of this band to stay readable
    band_vals = sub_band["dmmag"].dropna().to_numpy()
    if len(band_vals) > 0:
        y_lo, y_hi = np.nanpercentile(band_vals, [0.5, 99.5])
        pad = 0.15 * (y_hi - y_lo) if y_hi > y_lo else 1.0
        ax.set_ylim(y_lo - pad, y_hi + pad)

    ax.set_ylabel(r"$\Delta$mmag")
    ax.set_title(f"band {band!r}")
    ax.grid(True, alpha=0.3)

    if i // ncols_lc == 0:
        add_top_date_axis(ax, mjd_min_all, mjd_max_all)
    if i // ncols_lc == nrows_lc - 1:
        ax.set_xlabel("MJD")
    if i == 0:
        ax.legend(markerscale=2.5, fontsize=8, ncol=2, title="DDF", loc="upper right")

    ax.set_ylim(-YAXIS_MAX_MAG, YAXIS_MAX_MAG)

for j in range(n_bands_lc, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Per-object flux deviation from its median (mmag) vs time — color: band, marker: DDF",
    y=1.02,
)
plt.tight_layout()
savefig(fig, "mmag_deviation_vs_mjd_perband")
plt.show()